# 模型家族验证：训练、评分、保存与续跑

覆盖十种基础分类器、类别特征与显式多折搜索。需要安装 `hscredit[boost,tune]` 的兼容依赖。示例验证接口一致性，不把演示得分当作上线效果。

普通建模与单次验证请先看 `model_training_chain.ipynb`；本例的 `cv=3` 是显式选择。

## 路径和依赖

输出目录可在本单元格修改。SQLite Study 支持续跑；改变数据或配置时使用新的输出目录，避免混用历史试验。

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "hscredit").is_dir())
sys.path.insert(0, str(root))
input_path = Path(os.environ.get("HSCREDIT_EXAMPLE_INPUT", root / "examples/hscredit_yyp.xlsx"))
output = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT", root / "artifacts/model_family_validation"))
output.mkdir(parents=True, exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from hscredit import (CatBoost, DecisionTreeClassifier, ExtraTrees, GradientBoosting,
                      LightGBM, LogisticRegression, ModelTuner, NGBoost, RandomForest, SVM, XGBoost)
from hscredit.core.models._lifecycle import dependency_versions

## 1. 读取数据并拟合训练集预处理

In [ ]:
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
frame = pd.read_excel(input_path)
frame = frame.loc[frame["FPD"].isin([0, 1])].copy()
train, test = train_test_split(frame, test_size=0.2, random_state=42, stratify=frame["FPD"])
medians = train[features].median()
train[features] = train[features].fillna(medians)
test[features] = test[features].fillna(medians)
scaler = StandardScaler().fit(train[features])
from hscredit.utils import save_pickle

save_pickle({"标准化器": scaler, "特征顺序": features}, output / "preprocessor.joblib")
X_train = pd.DataFrame(scaler.transform(train[features]), columns=features, index=train.index)
X_test = pd.DataFrame(scaler.transform(test[features]), columns=features, index=test.index)
y_train, y_test = train["FPD"], test["FPD"]
training = X_train.assign(FPD=y_train)

## 2. 配置十种分类模型

In [ ]:
options = dict(random_state=42, target="FPD")
models = [
    LogisticRegression(max_iter=500, **options),
    RandomForest(n_estimators=20, n_jobs=1, **options),
    ExtraTrees(n_estimators=20, n_jobs=1, **options),
    GradientBoosting(n_estimators=20, **options),
    DecisionTreeClassifier(max_depth=3, **options),
    SVM(kernel="linear", **options),
    XGBoost(n_estimators=20, n_jobs=1, eval_metric=["auc", "ks"], early_stopping_rounds=3, **options),
    LightGBM(n_estimators=20, n_jobs=1, eval_metric=["auc", "ks"], early_stopping_rounds=3, **options),
    CatBoost(n_estimators=20, n_jobs=1, early_stopping_rounds=3, **options),
    NGBoost(n_estimators=20, n_jobs=1, base_max_depth=2, base_min_samples_leaf=20, **options),
]

## 3. 训练、评分与模型保存恢复

In [ ]:
results = []
for model in models:
    name = type(model).__name__
    model.fit(training)
    probabilities = model.predict_proba(X_test)
    scores = model.predict_score(X_test)
    restored = type(model).load(model.save(output / f"{name}.joblib"))
    np.testing.assert_allclose(restored.predict_proba(X_test.iloc[:, ::-1]), probabilities)
    np.testing.assert_allclose(restored.predict_score(X_test), scores)
    assert restored.training_summary_["状态"] == "完成"
    metrics = model.evaluate(X_test, y_test, metrics=["auc", "ks"])
    results.append({"模型": name, "训练样本": len(training), "测试样本": len(test), "状态": "通过", **metrics})
display(pd.DataFrame(results))

## 4. 验证类别特征输入

In [ ]:
# 类别字段按名称传入，标签列放在第一列也不会改变类别字段定位。
category_features = [*features, "商品类别"]
categorical = train[["FPD", *category_features]]
category_model = CatBoost(n_estimators=10, n_jobs=1, cat_features=["商品类别"], **options).fit(categorical)
assert category_model.predict_proba(test[category_features]).shape == (len(test), 2)

## 5. 使用多折搜索、保存与续跑

In [ ]:
import optuna

study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{(output / 'tuning.db').as_posix()}",
    study_name="模型工作流验证",
    load_if_exists=True,
)

def search(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 15),
    }

model = RandomForest(n_estimators=10, n_jobs=1, **options)
best = model.tune(
    training,
    search_space=search,
    study=study,
    cv=3,
    n_jobs=1,
    n_trials=2,
    show_progress_bar=False,
    artifact_dir=str(output / "trials"),
)
tuner = model.tuner
assert tuner.get_best_model() is best
assert tuner.get_oof_predictions()["预测概率"].notna().all()
restored_tuner = ModelTuner.load(tuner.save(output / "tuner.joblib"))
previous_trials = len(restored_tuner.study_.trials)
restored_tuner.fit(training, n_trials=1, show_progress_bar=False)
assert len(restored_tuner.study_.trials) == previous_trials + 1
assert restored_tuner.get_best_model().predict_score(X_test).shape == (len(test),)

## 6. 报告与验收结果

In [ ]:
# 原报告入口及逐轮曲线保留。
report = models[1].generate_report(X_train, y_train, X_test, y_test)
assert not report.summary().empty
payload = {
    "数据样本数": len(frame),
    "依赖版本": dependency_versions(),
    "模型验证": results,
    "类别特征验证": "通过",
    "调参保存续跑": "通过",
    "报告验证": "通过",
}
(output / "validation.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(pd.DataFrame(results).to_string(index=False))
print(f"验证结果与模型制品：{output.resolve()}")
result = payload
result